In [ ]:
# =========================================================
# CLEAN START — NO PROTOBUF OVERRIDE
# =========================================================
!pip install -q transformers accelerate bitsandbytes tqdm

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import pandas as pd
import csv
import time
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
logging.getLogger("transformers.generation.configuration_utils").setLevel(logging.ERROR)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
transformers.logging.set_verbosity_error()

print("GPU:", torch.cuda.get_device_name(0))


# =========================================================
# LOAD BOOLEAN EXPRESSIONS DATASET
# =========================================================
input_file = "/kaggle/input/formal-fallacies/formal_fallacies_perturbations.csv"
df = pd.read_csv(input_file)
df["id"] = df["id"].astype(int)

print("Rows:", len(df))


# =========================================================
# LOAD PARTIAL OUTPUT FOR RESUMING
# =========================================================
saved_output = "/kaggle/input/formal-fallacies/formal-fallacies_answers_qwen2.5.csv"

if os.path.isfile(saved_output):
    df_saved = pd.read_csv(saved_output)
    df_saved["id"] = df_saved["id"].astype(int)
    completed_ids = set(df_saved["id"])
    print("Completed:", len(completed_ids))
else:
    completed_ids = set()
    print("Starting fresh.")


df_remaining = df[~df["id"].isin(completed_ids)].reset_index(drop=True)

print("Remaining:", len(df_remaining))
print("Next ID:", df_remaining.iloc[0]["id"] if len(df_remaining)>0 else "DONE")


# =========================================================
# LOAD QWEN MODEL (FP16, GPU)
# =========================================================
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()


# =========================================================
# PROMPT TEMPLATE
# =========================================================
BASE_PROMPT = """Analyze the argument below. Think step by step using only the given premises.
Explain whether the conclusion must follow from the premises.

At the end, output the final answer in this exact format:
Final Answer: valid
or
Final Answer: invalid

Argument:
{QUESTION}

"""


# =========================================================
# GENERATION FUNCTION
# =========================================================
def generate_answer(prompt: str, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):]
    return decoded.strip()


# =========================================================
# PREP OUTPUT FILE
# =========================================================
output_file = "formal-fallacies_answers_qwen2.5.csv"
f = open(output_file, "a", newline="", encoding="utf-8")
writer = csv.writer(f)

if os.stat(output_file).st_size == 0:
    writer.writerow(["id","original_answer","lexical_answer","syntactic_answer","contextual_answer"])
    f.flush()


# =========================================================
# PROCESS REMAINING ROWS SAFELY
# =========================================================
for _, row in tqdm(df_remaining.iterrows(), total=len(df_remaining)):

    qid = row["id"]

    variants = {
        "original": row["original"],
        "lexical": row["lexical"],
        "syntactic": row["syntactic"],
        "contextual": row["contextual"],
    }

    answers = {}

    for key, question in variants.items():
        if not isinstance(question, str) or not question.strip():
            answers[f"{key}_answer"] = ""
            continue

        prompt = BASE_PROMPT.format(QUESTION=question)

        try:
            answers[f"{key}_answer"] = generate_answer(prompt)
        except Exception as e:
            answers[f"{key}_answer"] = f"ERROR: {e}"

        time.sleep(0.1)

    writer.writerow([
        qid,
        answers["original_answer"],
        answers["lexical_answer"],
        answers["syntactic_answer"],
        answers["contextual_answer"]
    ])
    f.flush()

f.close()

print("✔ DONE")


**generate missing answers**

In [ ]:
# =========================================================
# CLEAN START — NO PROTOBUF OVERRIDE
# =========================================================
!pip install -q transformers accelerate bitsandbytes tqdm

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import csv
import time
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
logging.getLogger("transformers.generation.configuration_utils").setLevel(logging.ERROR)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
transformers.logging.set_verbosity_error()

print("GPU:", torch.cuda.get_device_name(0))


# =========================================================
# LOAD ONLY MISSING QUESTIONS
# =========================================================
input_file = "/kaggle/input/formal-fallacies/formal-fallacies_missing_questions.csv"
df = pd.read_csv(input_file)
df["id"] = df["id"].astype(int)

print("Missing rows to process:", len(df))


# =========================================================
# LOAD QWEN MODEL
# =========================================================
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()


# =========================================================
# PROMPT TEMPLATE
# =========================================================
BASE_PROMPT = """Analyze the argument below. Think step by step using only the given premises.
Explain whether the conclusion must follow from the premises.

At the end, output the final answer in this exact format:
Final Answer: valid
or
Final Answer: invalid

Argument:
{QUESTION}

"""


# =========================================================
# GENERATION FUNCTION
# =========================================================
def generate_answer(prompt: str, max_new_tokens=1024):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):]
    return decoded.strip()


# =========================================================
# OUTPUT FILE (3 columns)
# =========================================================
output_file = "formal-fallacies_missing_answers.csv"
f = open(output_file, "w", newline="", encoding="utf-8")
writer = csv.writer(f)

# Write the required header
writer.writerow(["id", "column", "answer"])
f.flush()


# =========================================================
# PROCESS EACH MISSING QUESTION
# =========================================================
for _, row in tqdm(df.iterrows(), total=len(df)):

    qid = int(row["id"])
    col = row["column"]        # "original", "lexical", "syntactic", "contextual"
    question = row["question"]

    prompt = BASE_PROMPT.format(QUESTION=question)

    try:
        answer_text = generate_answer(prompt)
    except Exception as e:
        answer_text = f"ERROR: {e}"

    # Save a single row: id, column, answer
    writer.writerow([qid, col, answer_text])
    f.flush()

    time.sleep(0.1)

f.close()

print("✔ DONE — Missing answers saved to:", output_file)


In [ ]:
# =========================================================
# CLEAN START — NO PROTOBUF OVERRIDE
# =========================================================
!pip install -q transformers accelerate bitsandbytes tqdm

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import pandas as pd
import csv
import time
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
logging.getLogger("transformers.generation.configuration_utils").setLevel(logging.ERROR)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
transformers.logging.set_verbosity_error()

print("GPU:", torch.cuda.get_device_name(0))


# =========================================================
# LOAD BOOLEAN EXPRESSIONS DATASET
# =========================================================
input_file = "/kaggle/input/formal-fallacies/formal_fallacies_perturbations.csv"
df = pd.read_csv(input_file)
df["id"] = df["id"].astype(int)

print("Rows:", len(df))


# =========================================================
# IDs TO SKIP
# =========================================================
skip_ids = {
    3,6,7,16,27,39,40,41,44,47,52,54,61,62,64,66,68,71,72,73,75,78,80,84,85,92,99,102,
    107,109,115,116,123,124,130,135,141,144,147,153,155,160,167,171,173,175,177,184,
    189,194,197,199,215,218,239,244
}


# =========================================================
# LOAD PARTIAL OUTPUT FOR RESUMING
# =========================================================
saved_output = "/kaggle/input/formal-fallacies"

if os.path.isfile(saved_output):
    df_saved = pd.read_csv(saved_output)
    df_saved["id"] = df_saved["id"].astype(int)
    completed_ids = set(df_saved["id"])
    print("Completed:", len(completed_ids))
else:
    completed_ids = set()
    print("Starting fresh.")


# Filter rows that are NOT completed and NOT skipped
df_remaining = df[
    (~df["id"].isin(completed_ids)) & (~df["id"].isin(skip_ids))
].reset_index(drop=True)

print("Remaining:", len(df_remaining))
print("Next ID:", df_remaining.iloc[0]["id"] if len(df_remaining)>0 else "DONE")


# =========================================================
# LOAD QWEN MODEL (FP16, GPU)
# =========================================================
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()


# =========================================================
# PROMPT TEMPLATE (Improved CoT + No LaTeX)
# =========================================================
BASE_PROMPT = """
Analyze the argument below. Follow these steps carefully:

1. Restate the premises in simple plain English.
2. Restate the conclusion in simple plain English.
3. Explain step by step whether the conclusion logically follows from the premises.
4. Use only verbal reasoning and natural language.
5.- You are NOT allowed to use LaTeX commands or formatting. 
  Do NOT use: $, \[, \], \(, \), \\begin, \\end, \\forall, \\exists, \\land, \\lor, \\neg, or any other LaTeX syntax.

At the end, output the final answer on its own line in this exact format:
Final Answer: valid
or
Final Answer: invalid

Argument:
{QUESTION}
"""


# =========================================================
# GENERATION FUNCTION
# =========================================================
def generate_answer(prompt: str, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):]
    return decoded.strip()


# =========================================================
# PREP OUTPUT FILE
# =========================================================
# IMPORTANT: You must save to /kaggle/working (writeable),
# NOT to /kaggle/input (read-only).
output_file = "/kaggle/working/formal-fallacies_answers_qwen2.5.csv"

file_exists = os.path.isfile(output_file)
f = open(output_file, "a", newline="", encoding="utf-8")
writer = csv.writer(f)

if not file_exists:
    writer.writerow(["id","original_answer","lexical_answer","syntactic_answer","contextual_answer"])
    f.flush()


# =========================================================
# PROCESS REMAINING ROWS SAFELY
# =========================================================
for _, row in tqdm(df_remaining.iterrows(), total=len(df_remaining)):

    qid = row["id"]

    variants = {
        "original": row["original"],
        "lexical": row["lexical"],
        "syntactic": row["syntactic"],
        "contextual": row["contextual"],
    }

    answers = {}

    for key, question in variants.items():
        if not isinstance(question, str) or not question.strip():
            answers[f"{key}_answer"] = ""
            continue

        prompt = BASE_PROMPT.format(QUESTION=question)

        try:
            answers[f"{key}_answer"] = generate_answer(prompt)
        except Exception as e:
            answers[f"{key}_answer"] = f"ERROR: {e}"

        time.sleep(0.1)

    writer.writerow([
        qid,
        answers["original_answer"],
        answers["lexical_answer"],
        answers["syntactic_answer"],
        answers["contextual_answer"]
    ])
    f.flush()

f.close()

print("✔ DONE — output saved to /kaggle/working/")


In [ ]:
!git clone https://github.com/eladsegal/strategyqa.git


In [ ]:
import json
import os

# Path to the original dataset
train_path = "strategyqa/data/strategyqa/train.json"

# Load full train set
with open(train_path, "r") as f:
    train_data = json.load(f)

print("Total examples in train:", len(train_data))

# Take exactly 350 examples
subset_350 = train_data[:350]

print("Subset size:", len(subset_350))


In [ ]:
output_path = "strategyqa_350.json"

with open(output_path, "w") as f:
    json.dump(subset_350, f, indent=2)

print("Saved:", output_path)


In [ ]:
import pandas as pd

# Convert the subset (list of dicts) into a DataFrame
df = pd.DataFrame(subset_350)

# Save as CSV
output_csv_path = "strategyqa_350.csv"
df.to_csv(output_csv_path, index=False)

print("Saved CSV:", output_csv_path)
print(df.head())


In [ ]:
!pip install -q protobuf==3.20.3


**strategyQA new try**

In [ ]:
!pip -q install transformers accelerate sentencepiece tqdm

import os
import csv
import time
import pandas as pd
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

# ---------------------------
# LOAD Qwen2.5-7B-Instruct
#   (for more speed, you can switch to:
#    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct")
# ---------------------------
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print("🚀 Loading Qwen2.5-7B-Instruct ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",          # keep auto-sharding to avoid OOM
    torch_dtype="auto"
)
model.eval()

# ---------------------------
# CONFIG
# ---------------------------
INPUT_FILE = "/kaggle/input/strategyqa/strategyqa_350_perturbed.csv"
OUTPUT_FILE = "/kaggle/working/strategyqa_350_perturbed_qwen_rationales.csv"

SLEEP_BETWEEN_CALLS = 0.0   # no delay

# ---------------------------
# PROMPT TEMPLATE
# ---------------------------
PROMPT_TEMPLATE = """You are an expert reasoner. You must answer the question with either True or False.

Question:
{question}

Instructions:
1. Think step by step and explain your reasoning in detail.
2. Use any relevant world knowledge needed to answer the question.
3. At the end of your reasoning, give the final answer as either True or False.

Important:
- At the end, use this exact format on a separate line:
  Final Answer: True
or
  Final Answer: False

Now, reason carefully and then provide your final answer.
"""

# ---------------------------
# QWEN BATCHED CALL FUNCTION
# ---------------------------
def get_qwen_rationales_batch(questions):
    """
    Generate CoT rationales + final answers for a batch of questions
    using Qwen2.5-7B-Instruct.

    questions: list[str] (e.g., [orig, lex, syn, ind])
    returns: list[str] same length
    """
    prompts = [PROMPT_TEMPLATE.format(question=q) for q in questions]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=524,           # ↓ smaller than 512 => faster
            temperature=0.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    if SLEEP_BETWEEN_CALLS > 0:
        time.sleep(SLEEP_BETWEEN_CALLS)

    return texts  # list of strings


# ---------------------------
# MAIN SCRIPT
# ---------------------------




**e-SNLI**

In [1]:
!pip install -q --force-reinstall protobuf==3.20.3
!pip install -q transformers accelerate

import warnings
warnings.filterwarnings("ignore")

import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.9 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptograp

In [ ]:
# ============================================================
# FIX ENV + DEPENDENCIES
# ============================================================
import logging
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import csv
from tqdm.auto import tqdm

# Suppress warnings
logging.getLogger("transformers").setLevel(logging.ERROR)

# ============================================================
# PATHS
# ============================================================
INPUT_PATH = "/kaggle/input/e-snli/esnli_800_with_perturbations.csv"
OUTPUT_PATH = "/kaggle/working/qwen_esnli_rationales.csv"

# ============================================================
# MODEL CONFIGURATION
# ============================================================
model_name = "Qwen/Qwen2.5-7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

# ============================================================
# SYSTEM PROMPT (Instructions + Examples)
# ============================================================
# We separate the static instructions from the dynamic user input.
# This helps the model distinguish context from the current task.

SYSTEM_PROMPT = """
You are an expert NLI reasoning engine. Your task is to classify the logical relationship between a Premise and a Hypothesis.

### INSTRUCTIONS (STRICT):

1. **FUNDAMENTAL RULE**:
   - You must assume the Premise is an indisputable fact in a closed logical world.
   - **Do NOT** evaluate realism. Base your reasoning ONLY on the logical consequences of the premise being true.

2. Determine the correct label:
   - **ENTAILMENT**: The hypothesis MUST be true given the premise.
   - **CONTRADICTION**: The hypothesis CANNOT be true given the premise.
   - **NEUTRAL**: The hypothesis MIGHT be true, but is not guaranteed.
     *(Self-Check: If a logically consistent scenario exists where Premise=True and Hypothesis=False, choose NEUTRAL.)*

3. Write a concise "Chain of Thought" (1–3 sentences).
   - **Focus on Definitions:** Analyze the concepts abstractly (e.g., "X implies Y"). Do NOT retell the story or describe the specific scene.
   - **Deterministic Language:** Avoid hedging ("likely", "possibly", "could"). Use categorical terms ("guarantees", "excludes", "does not ensure").
   - **No Meta-Talk:** Do NOT use phrases like "The premise states..." or "According to the text...".

### EXAMPLES (Imitate this exact abstract style):

Premise: A soccer player is running across the field.
Hypothesis: A person is moving.
Chain of Thought: The category "soccer player" falls under the category "person", and "running" is a specific type of "movement". Therefore, the premise logically entails the hypothesis.
Final Answer: ENTAILMENT

Premise: A black dog is sleeping on the couch.
Hypothesis: A dog is running outside.
Chain of Thought: The states of "sleeping" and "running" are mutually exclusive; an entity cannot perform both simultaneously. The premise excludes the hypothesis.
Final Answer: CONTRADICTION

Premise: A woman is buying vegetables at the market.
Hypothesis: The woman is preparing dinner.
Chain of Thought: The act of "buying vegetables" does not logically compel "preparing dinner"; acquisition is distinct from preparation. Since the outcome is not guaranteed, the relationship is inconclusive.
Final Answer: NEUTRAL

---

Now solve this one:

Premise: {premise}
Hypothesis: {hypothesis}
Chain of Thought:
"""

# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_rationales_batch(premise, hypothesis):
    """
    Constructs the prompt using the official Chat Template.
    This ensures the model sees <|im_start|> and <|im_end|> tokens,
    preventing it from rambling endlessly.
    """
    
    # We construct the user message clearly
    user_content = f"Premise: {premise}\nHypothesis: {hypothesis}\nChain of Thought:"
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]
    
    # 1. Apply Chat Template (Adds special tokens automatically)
    text_input = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text_input], return_tensors="pt").to(device)

    # 2. Define specific Qwen terminators (Stop tokens)
    # This includes <|im_end|> (id 151645) and <|endoftext|> (id 151643)
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|im_end|>"),
        tokenizer.convert_tokens_to_ids("<|endoftext|>")
    ]

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=128,  # Reduced: We expect concise answers
        do_sample=False,     # Deterministic
        eos_token_id=terminators, # Explicitly tell it to stop
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode only the new tokens (removing the prompt)
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()

# ============================================================
# MAIN EXECUTION
# ============================================================

# Load Data
df = pd.read_csv(INPUT_PATH)

# Initialize Output File
with open(OUTPUT_PATH, "w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "rationale_original", "rationale_perturbation_1_lexical", 
                     "rationale_perturbation_2_syntactic", "rationale_perturbation_3_pragmatic"])

print("🚀 Starting generation with fixed Chat Template logic...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    premise = row["Premise"]
    
    hypotheses = {
        "original": row["Hypothesis"],
        "lexical": row["perturbation_1_lexical"],
        "syntactic": row["perturbation_2_syntactic"],
        "pragmatic": row["perturbation_3_pragmatic"],
    }
    
    results = {}
    
    for key, hyp in hypotheses.items():
        # Call the fixed generator
        output = generate_rationales_batch(premise, hyp)
        
        # Cleanup: Sometimes Qwen repeats the "Chain of Thought:" header if we put it in the user prompt.
        # This is a cosmetic cleanup.
        if output.startswith("Chain of Thought:"):
            output = output.replace("Chain of Thought:", "", 1).strip()
            
        results[key] = output
        
    # Write immediately
    with open(OUTPUT_PATH, "a", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            row["id"],
            results["original"],
            results["lexical"],
            results["syntactic"],
            results["pragmatic"]
        ])

print(f"✅ Done! Output saved to: {OUTPUT_PATH}")


Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

🚀 Starting generation with fixed Chat Template logic...


Processing:   0%|          | 0/800 [00:00<?, ?it/s]

In [2]:
# ============================================================
# FIX ENV + DEPENDENCIES
# ============================================================
import logging
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import csv
import os
from tqdm.auto import tqdm

# Suppress warnings
logging.getLogger("transformers").setLevel(logging.ERROR)

# ============================================================
# PATHS
# ============================================================
INPUT_PATH = "/kaggle/input/e-snli/esnli_800_with_perturbations.csv"
OUTPUT_PATH = "/kaggle/working/qwen_esnli_rationales.csv"
RESUME_PATH = "/kaggle/input/e-snli/qwen_esnli_rationales.csv" # Path to your interrupted file

# ============================================================
# MODEL CONFIGURATION
# ============================================================
model_name = "Qwen/Qwen2.5-7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

# ============================================================
# SYSTEM PROMPT (Instructions + Examples)
# ============================================================
SYSTEM_PROMPT =  """
You are an expert NLI reasoning engine. Your task is to classify the logical relationship between a Premise and a Hypothesis.

### INSTRUCTIONS (STRICT):

1. **FUNDAMENTAL RULE**:
   - You must assume the Premise is an indisputable fact in a closed logical world.
   - **Do NOT** evaluate realism. Base your reasoning ONLY on the logical consequences of the premise being true.

2. Determine the correct label:
   - **ENTAILMENT**: The hypothesis MUST be true given the premise.
   - **CONTRADICTION**: The hypothesis CANNOT be true given the premise.
   - **NEUTRAL**: The hypothesis MIGHT be true, but is not guaranteed.
     *(Self-Check: If a logically consistent scenario exists where Premise=True and Hypothesis=False, choose NEUTRAL.)*

3. Write a concise "Chain of Thought" (1–3 sentences).
   - **Focus on Definitions:** Analyze the concepts abstractly (e.g., "X implies Y"). Do NOT retell the story or describe the specific scene.
   - **Deterministic Language:** Avoid hedging ("likely", "possibly", "could"). Use categorical terms ("guarantees", "excludes", "does not ensure").
   - **No Meta-Talk:** Do NOT use phrases like "The premise states..." or "According to the text...".

### EXAMPLES (Imitate this exact abstract style):

Premise: A soccer player is running across the field.
Hypothesis: A person is moving.
Chain of Thought: The category "soccer player" falls under the category "person", and "running" is a specific type of "movement". Therefore, the premise logically entails the hypothesis.
Final Answer: ENTAILMENT

Premise: A black dog is sleeping on the couch.
Hypothesis: A dog is running outside.
Chain of Thought: The states of "sleeping" and "running" are mutually exclusive; an entity cannot perform both simultaneously. The premise excludes the hypothesis.
Final Answer: CONTRADICTION

Premise: A woman is buying vegetables at the market.
Hypothesis: The woman is preparing dinner.
Chain of Thought: The act of "buying vegetables" does not logically compel "preparing dinner"; acquisition is distinct from preparation. Since the outcome is not guaranteed, the relationship is inconclusive.
Final Answer: NEUTRAL

---

Now solve this one:

Premise: {premise}
Hypothesis: {hypothesis}
Chain of Thought:
"""


# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_rationales_batch(premise, hypothesis):
    """
    Constructs the prompt using the official Chat Template.
    """
    user_content = f"Premise: {premise}\nHypothesis: {hypothesis}\nChain of Thought:"
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]
    
    text_input = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text_input], return_tensors="pt").to(device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|im_end|>"),
        tokenizer.convert_tokens_to_ids("<|endoftext|>")
    ]

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=128,
        do_sample=False,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()

# ============================================================
# MAIN EXECUTION (WITH RESUME LOGIC)
# ============================================================

# 1. Load the main input data
df = pd.read_csv(INPUT_PATH)
processed_ids = set()

# 2. Check for existing progress file to resume
if os.path.exists(RESUME_PATH):
    print(f"🔄 Found interrupted output file at: {RESUME_PATH}")
    
    # Read the existing processed rows
    try:
        df_existing = pd.read_csv(RESUME_PATH)
        processed_ids = set(df_existing["id"].tolist())
        print(f"📊 Resume stats: {len(processed_ids)} rows already processed.")
        
        # Write existing data to the NEW output path so we don't lose it
        df_existing.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
        print(f"✅ Copied existing data to {OUTPUT_PATH}. Will append new rows there.")
        
    except Exception as e:
        print(f"⚠️ Error reading resume file: {e}. Starting from scratch.")
        # Create fresh file if reading failed
        with open(OUTPUT_PATH, "w", newline='', encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["id", "rationale_original", "rationale_perturbation_1_lexical", 
                             "rationale_perturbation_2_syntactic", "rationale_perturbation_3_pragmatic"])
else:
    print("🆕 No previous run found. Starting from scratch.")
    # Create fresh file
    with open(OUTPUT_PATH, "w", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "rationale_original", "rationale_perturbation_1_lexical", 
                         "rationale_perturbation_2_syntactic", "rationale_perturbation_3_pragmatic"])

# 3. Filter the dataframe to only include UNPROCESSED rows
df_to_process = df[~df["id"].isin(processed_ids)]
print(f"🚀 Rows remaining to process: {len(df_to_process)} (out of {len(df)})")

# 4. Start the loop
for idx, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Processing Remaining"):
    premise = row["Premise"]
    
    hypotheses = {
        "original": row["Hypothesis"],
        "lexical": row["perturbation_1_lexical"],
        "syntactic": row["perturbation_2_syntactic"],
        "pragmatic": row["perturbation_3_pragmatic"],
    }
    
    results = {}
    
    for key, hyp in hypotheses.items():
        output = generate_rationales_batch(premise, hyp)
        
        if output.startswith("Chain of Thought:"):
            output = output.replace("Chain of Thought:", "", 1).strip()
            
        results[key] = output
        
    # Append immediately
    with open(OUTPUT_PATH, "a", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            row["id"],
            results["original"],
            results["lexical"],
            results["syntactic"],
            results["pragmatic"]
        ])

print(f"✅ All DONE! Final complete file saved to: {OUTPUT_PATH}")

Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

🔄 Found interrupted output file at: /kaggle/input/e-snli/qwen_esnli_rationales.csv
📊 Resume stats: 34 rows already processed.
✅ Copied existing data to /kaggle/working/qwen_esnli_rationales.csv. Will append new rows there.
🚀 Rows remaining to process: 766 (out of 800)


Processing Remaining:   0%|          | 0/766 [00:00<?, ?it/s]

✅ All DONE! Final complete file saved to: /kaggle/working/qwen_esnli_rationales.csv
